# DeltaNet v2: Fused Linear Attention on Turing GPUs
### Interactive Tutorial & Demonstration Notebook

Inspired by NVIDIA's paper [**Gated DeltaNet v2** (arXiv:2605.22791)](https://arxiv.org/pdf/2605.22791), this implementation decouples the Key and Value gating into independent gates to optimize autoregressive inference. While most modern open-source repositories focus exclusively on A100/H100+ hardware, this implementation is tailored specifically for **NVIDIA Turing SM75** GPUs (Tesla T4 on Google Colab/Kaggle and consumer RTX 20-series) to make linear attention accessible for small-scale experimentation and learning.

> *Note: Source code and annotations are provided for reference. Built with an agentic workflow using strict design constraints to ensure logical correctness, register pressure control, and efficient on-chip memory staging.*

This tutorial covers:
1. Installation & Environment Check.
2. Constructing a `DeltaBlock` with Grouped Query Attention (GQA).
3. Variable-length packed sequence training with fused CUDA autograd.
4. Fast single-step ($L = 1$) autoregressive generation.
5. Numerical consistency check against pure PyTorch eager execution.
6. Acceleration with `torch.compile`.

## 1. Requirements & Installation
Requirements: `torch >= 2.0.0`, `setuptools >= 61.0`, `wheel`, and `ninja` (for accelerated compilation).
CUDA Toolkit `>= 11.8` or `12.x` and a C++17 compiler (`g++ >= 9.0`) are required.

In [ ]:
# Install ninja for faster parallel compilation (recommended):
# !pip install ninja

# In Google Colab or Kaggle, install directly from GitHub:
# !pip install git+https://github.com/CBG6682/Deltanet.git

# If running locally from this directory:
!pip install -e .


## 2. Verify GPU & Environment
Let us verify CUDA device availability and check that the extension is accessible.

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Compute Capability: {torch.cuda.get_device_capability(0)}")


## 3. Instantiate DeltaBlock
`DeltaBlock` encapsulates the linear recurrence with WY chunking, decay gating ($\alpha$), erase gating ($\beta$), write gating ($W$), and normalization scaling ($\gamma$).

In [ ]:
from deltanet_v2 import DeltaBlock

device = "cuda" if torch.cuda.is_available() else "cpu"

d_model = 768
num_q_heads = 8
num_kv_heads = 4  # GQA: 8 query heads share 4 KV heads (group size = 2)

model = DeltaBlock(
    d_model=d_model,
    num_q_heads=num_q_heads,
    num_kv_heads=num_kv_heads,
    d_k=64,
    d_v=64,
    chunk_size=64,
    hidden_dim=d_model * 4,
).to(device=device, dtype=torch.float16)

print(model)


## 4. Training Pass (Variable-Length Packed Sequences)
In training, sequences of different lengths (e.g. 512, 1024) are concatenated into a packed 1D token stream `total_L = 1536`, indexed by `cu_seqlens = [0, 512, 1536]`.
`use_kernel=True` invokes the fused CuTe CUDA forward and backward kernels.

In [ ]:
model.train()

# Construct 2 sequences of lengths 512 and 1024
cu_seqlens = torch.tensor([0, 512, 1536], dtype=torch.int32, device=device)
total_L = 1536

x = torch.randn(total_L, d_model, device=device, dtype=torch.float16, requires_grad=True)

# Forward pass
out, final_states = model(x, cu_seqlens=cu_seqlens, use_kernel=True)
print(f"Output shape: {out.shape}")
print(f"Final recurrent states shape: {final_states.shape}")

# Backward pass
loss = out.mean()
loss.backward()
print(f"Gradient on input x shape: {x.grad.shape}")
print(f"Gradient norm: {x.grad.norm().item():.4f}")


## 5. Autoregressive Generation ($L = 1$ Decode Step)
During generation, tokens are emitted one-by-one ($L = 1$). `DeltaBlock` uses a dedicated decode kernel that updates the recurrent state in place and produces attention output at extreme throughput.

In [ ]:
import time

model.eval()
batch_size = 128
num_decode_steps = 10

# Recurrent state shape: (batch_size, num_kv_heads, d_v, d_k)
state = torch.zeros(batch_size, num_kv_heads, 64, 64, device=device, dtype=torch.float16)

latencies = []
with torch.no_grad():
    for step in range(num_decode_steps):
        token = torch.randn(batch_size, 1, d_model, device=device, dtype=torch.float16)
        if device == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        
        y, state = model(token, past_states=state, use_kernel=True)
        
        if device == "cuda":
            torch.cuda.synchronize()
        t1 = time.perf_counter()
        latencies.append((t1 - t0) * 1000)

avg_ms = sum(latencies[2:]) / len(latencies[2:])  # warm-up discounted
throughput = (batch_size / avg_ms) * 1000
print(f"Avg step latency: {avg_ms:.2f} ms")
print(f"Throughput: {throughput:.1f} tokens/second")


## 6. Numerical Precision Verification
Compare the output of the high-performance CUDA kernel against the pure PyTorch baseline implementation to verify precision.

In [ ]:
model.eval()

# Single sequence of length 256 for testing
test_len = 256
x_test = torch.randn(1, test_len, d_model, device=device, dtype=torch.float16)
cu_test = torch.tensor([0, test_len], dtype=torch.int32, device=device)

with torch.no_grad():
    # 1. PyTorch baseline (use_kernel=False)
    out_ref, state_ref = model(x_test, use_kernel=False)
    
    # 2. CUDA kernel (use_kernel=True, packed 1D format)
    x_packed = x_test.squeeze(0)
    out_kernel, state_kernel = model(x_packed, cu_seqlens=cu_test, use_kernel=True)
    out_kernel = out_kernel.unsqueeze(0)

# Compute cosine similarity and max relative difference
cos_sim = torch.nn.functional.cosine_similarity(out_ref.flatten(), out_kernel.flatten(), dim=0)
max_diff = torch.max(torch.abs(out_ref - out_kernel)).item()

print(f"Cosine Similarity: {cos_sim.item():.6f}")
print(f"Max Absolute Difference: {max_diff:.6f}")
assert cos_sim.item() > 0.9999, "Numerical mismatch between CUDA kernel and PyTorch reference!"
print("Numerical Check: PASSED!")


## 7. Acceleration with `torch.compile`
DeltaNet v2 kernels register fake/meta tensors with the PyTorch Dispatcher, enabling full graph capture under `torch.compile` without graph breaks.

In [ ]:
compiled_model = torch.compile(model)

with torch.no_grad():
    out_compiled, _ = compiled_model(x_packed, cu_seqlens=cu_test, use_kernel=True)

print("Compiled execution successful! Output shape:", out_compiled.shape)
